In [2]:
import os
import numpy as np
import cv2
from scipy.io import savemat
from natsort import natsorted
import h5py
from scipy.io import savemat, loadmat
from scipy import interpolate


def interpolate_HS_Cube(new_channels_nm, hs_cube, hs_bands):
    # Throw an error if we try to extrapolate
    if (min(new_channels_nm) < min(hs_bands) - 1) or (
        max(new_channels_nm) > max(hs_bands) + 1
    ):
        raise ValueError(
            f"In generator, extrapoaltion of the ARAD dataset outside of measurement data is not allowed: {min(hs_bands)}-{max(hs_bands)}"
        )

    interpfun = interpolate.interp1d(
        hs_bands,
        hs_cube,
        axis=-1,
        kind="linear",
        assume_sorted=True,
        fill_value="extrapolate",
        bounds_error=False,
    )
    resampled = interpfun(new_channels_nm)

    return resampled


In [ ]:
## Resave the original CAVE release to a more convenient formatting
## Format for scip.io loadmat and savemat since its fast
## Resaved as float32 
dtype = np.float32
root_dir = '/home/deanhazineh/ssd2tb/CAVE_Dataset/complete_ms_data/'
output_dir = os.path.join(root_dir, "./datasets/CAVE_repackaged/", 'HSI')
os.makedirs(output_dir, exist_ok=True)

for item_name in os.listdir(root_dir):
    item_path = os.path.join(root_dir, item_name)
    if os.path.isdir(item_path):
        item_subdir = os.path.join(item_path, item_name)
        if os.path.isdir(item_subdir):

            # Collect all PNG files in the subdirectory
            # This is the folder structure the original dataset is distributed by authors
            png_files = [f for f in os.listdir(item_subdir) if f.endswith('.png')]
            png_files = natsorted(png_files)
            images = []
            for png_file in png_files:
                img_path = os.path.join(item_subdir, png_file)
                img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
                if img.ndim == 3 and img.shape[2] == 4:  # Check if the image has 4 channels (RGBA)
                    img = cv2.cvtColor(img, cv2.COLOR_RGBA2GRAY)  # Convert to grayscale
                images.append(img)
            hsi_data = np.stack(images, axis=-1).astype(dtype)       
            mat_filename = os.path.join(output_dir, f'CAVE_train_{item_name}.mat')
            savemat(mat_filename, {"hsi": hsi_data})


In [5]:
# For CASSI Rendering on the TSA Benchmark Challenge, we need to resave the cubes with 256x256 chunking and float16
# We will load these datacubes, grab random 256x256 chunks, render the measurements, and then train on patches
# This is the same thing we did for the KAIST dataset
datpath = f"./datasets/CAVE_repackaged/"
savepath = f"./datasets/CASSI_Dataset_450_650/"
fnames = natsorted(os.listdir(datpath))
interp_ch = np.linspace(450, 650, 28)
ch = np.linspace(400, 700, 31)

for f in fnames:
    datfile = os.path.join(datpath, f)
    hsi = loadmat(datfile)["hsi"].astype(np.float32)
    hsi = hsi/hsi.max()

    hsi = np.clip(interpolate_HS_Cube(interp_ch, hsi, ch),0,1).astype(np.float16)
    hdf5_file_path = savepath+f'CAVE_{os.path.splitext(f)[0]}.h5'
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('hsi', data=hsi, chunks=(256, 256, 28), dtype='float16')


In [3]:
# Similar to other versions, save a copy of the HSI in a common range 420 to 700 for unconditioned mixed set training
datpath = f"./datasets/CAVE_repackaged/"
savepath = f"./datasets/HSI_multiset_420_700/"
fnames = natsorted(os.listdir(datpath))
interp_ch = np.arange(420, 710, 10)
ch = np.linspace(400, 700, 31)

for f in fnames:
    datfile = os.path.join(datpath, f)
    hsi = loadmat(datfile)["hsi"].astype(np.float32)
    hsi = hsi/hsi.max()

    hsi = np.clip(interpolate_HS_Cube(interp_ch, hsi, ch),0,1).astype(np.float16)
    hdf5_file_path = savepath+f'CAVE_{os.path.splitext(f)[0]}.h5'
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('hsi', data=hsi, chunks=(256, 256, 29), dtype='float16')
